# 10 · 规则引擎模块（Rules）功能演示

演示 Rule 表达式（含中文列名）、与/或/非/异或组合、规则报告(命中/坏率/Lift)、多逾期标签报告与表达式优化。

In [1]:
import warnings, os
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import hscredit

# 路径约定：从 notebooks/ 目录运行，数据在 ../examples，产物输出到 model_report/
DATA = os.path.join("..", "examples", "hscredit_yyp.xlsx")
if not os.path.exists(DATA):
    DATA = os.path.join("examples", "hscredit_yyp.xlsx")
OUT = "model_report"
os.makedirs(OUT, exist_ok=True)

df = pd.read_excel(DATA)
df["放款时间"] = pd.to_datetime(df["放款时间"])
y = df["FPD"].astype(int)
NUM_FEATURES = ["珊瑚92", "青云24", "衡枢鉴真分老客版", "占信V3", "天创小额网贷分", "近六个月非银多头机构数"]
CAT_FEATURE = "商品类别"
print("数据形状:", df.shape)
print("坏样本率: {:.4f}".format(y.mean()))
df.head()

数据形状: (970, 18)
坏样本率: 0.1402


,客户编号,放款时间,放款金额,商品类别,MOB1,CURRENT_DPD,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,近六个月非银多头机构数,手机号近一个月非银多头机构数,身份证近一个月非银多头机构数,衡枢鉴真分老客版,FPD
0,1985945640026276096,2026-02-03,1399,礼包,0,0,NaN,NaN,NaN,656,NaN,NaN,630,51,15,15,0.0242,0
1,1985972188268592896,2026-02-04,1399,礼包,0,0,NaN,NaN,NaN,565,NaN,NaN,583,56,6,18,0.0492,0
2,1986034700861140992,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,708,NaN,NaN,764,68,17,20,0.0546,0
3,1986264852923760896,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,555,NaN,NaN,712,45,15,15,0.0899,0
4,1986265696509906944,2026-01-26,1399,礼包,0,0,NaN,NaN,NaN,581,NaN,NaN,641,67,32,32,0.0678,0


## 1. Rule 定义与预测（支持中文列名，反引号包裹）

In [2]:
from hscredit.core.rules import Rule, get_columns_from_query, optimize_expr, beautify_expr

r1 = Rule("`衡枢鉴真分老客版` < 600", name="低分规则")
r2 = Rule("`青云24` > 500", name="青云高分")
r3 = Rule("`近六个月非银多头机构数` >= 5", name="多头机构多")

mask = r1.predict(df)
print('命中样本数:', int(mask.sum()), '/', len(df))
print('解析出的列:', get_columns_from_query("`衡枢鉴真分老客版` < 600 & `青云24` > 500"))
df.loc[mask, ['衡枢鉴真分老客版','FPD']].head()

命中样本数: 970 / 970
解析出的列: ['衡枢鉴真分老客版', '青云24']


,衡枢鉴真分老客版,FPD
0,0.0242,0
1,0.0492,0
2,0.0546,0
3,0.0899,0
4,0.0678,0


## 2. 规则组合：与(&) / 或(|) / 非(~) / 异或(^)

In [3]:
combos = {
    'r1 & r2': (r1 & r2), 'r1 | r2': (r1 | r2), '~r1': (~r1),
    'r1 ^ r2 (异或)': (r1 ^ r2),
}
rows = []
for name, rule in combos.items():
    hit = rule.predict(df).astype(bool)
    rows.append({'规则组合': name, '命中数': int(hit.sum()), '命中坏率': round(float(y[hit.values].mean()) if hit.sum()>0 else 0, 4)})
pd.DataFrame(rows)

,规则组合,命中数,命中坏率
0,r1 & r2,919,0.1393
1,r1 | r2,970,0.1402
2,~r1,0,0.0000
3,r1 ^ r2 (异或),51,0.1569


## 3. 规则报告 Rule.report（命中率 / 坏率 / Lift，中文指标）

In [4]:
df_t = df.copy(); df_t['target'] = y.values
report = r1.report(df_t, target='target', amount='放款金额')
report

,规则分类,指标名称,分箱,样本总数,样本占比,好样本数,好样本占比,坏样本数,坏样本占比,坏样本率,LIFT值,坏账改善,风险拒绝比,准确率,精确率,召回率,F1分数
0,验证规则,`衡枢鉴真分老客版` < 600,命中,4087203,1.0000,3506592,1.0000,580611,1.0000,0.1421,1.0000,1.0000,1.0000,0.1402,0.1402,1.0000,0.2459
1,验证规则,`衡枢鉴真分老客版` < 600,未命中,0,0.0000,0,0.0000,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.8598,0.0000,0.0000,0.0000


## 4. 多逾期标签规则报告（overdue / dpds）

In [5]:
r1.report(df, overdue=['MOB1'], dpds=[7, 3, 0])

分箱详情                                    MOB1 7+                            ... MOB1 0+                                                               
   规则分类              指标名称   分箱 样本总数   样本占比    好样本数  好样本占比 坏样本数  坏样本占比   坏样本率  ...    坏样本数  坏样本占比   坏样本率  LIFT值   坏账改善  风险拒绝比    准确率    精确率    召回率   F1分数
0  验证规则  `衡枢鉴真分老客版` < 600   命中  970 1.0000     825 1.0000  145 1.0000 0.1495  ...     199 1.0000 0.2052 1.0000 1.0000 1.0000 0.2052 0.2052 1.0000 0.3405
1  验证规则  `衡枢鉴真分老客版` < 600  未命中    0 0.0000       0 0.0000    0 0.0000 0.0000  ...       0 0.0000 0.0000 0.0000 0.0000 0.0000 0.7948 0.0000 0.0000 0.0000

[2 rows x 41 columns]

## 5. 规则过滤与表达式优化/美化

In [6]:
filtered = r1.filter(df)
print('过滤后样本数:', len(filtered))
print('优化:', optimize_expr("(`青云24` > 500) | (`青云24` > 500)"))
print('美化:', beautify_expr("`衡枢鉴真分老客版`<600&`青云24`>500"))

过滤后样本数: 970
优化: (`青云24` > 500) | (`青云24` > 500)
美化: `衡枢鉴真分老客版`<600&`青云24`>500


## 6. 规则报告导出 Excel

In [7]:
(r1 & r3).report(df_t, target='target').to_excel(f"{OUT}/10_rules_report.xlsx", index=False)
print('已保存规则报告')

已保存规则报告
